In [6]:
import pandas as pd
from itertools import product

excel_file = "resultados.xlsx"

architectures = [
    [32], [64], [128], [264],
    [8, 4], [16, 8], [32, 16], [64, 32], [128, 64], [264, 128],
    [16, 8, 4], [32, 16, 8], [128, 64, 32], [264, 128, 64]
]

r_values = [0.01, 0.5, 0.9]

N_MODELS = 5  # número de seeds

df = pd.read_excel(excel_file)

comb = list(product(architectures, r_values))

for i, (arch, r) in enumerate(comb):
    
    start = i * N_MODELS
    end = start + N_MODELS
    
    df.loc[start:end-1, "Neurons"] = str(arch)
    df.loc[start:end-1, "r"] = r

df.to_excel("resultados.xlsx", index=False)

In [7]:
import numpy as np

TARGETS = ["Y"]

SETS = [
    "Train_1",
    "Train_2",
    "Val",
    "Test_1",
    "Test_2",
    "Test_3"
]

results = pd.read_excel("resultados.xlsx")


In [8]:
best_models_tables = {}

summary_all = []     # estatísticas com todos
summary_top10 = []   # estatísticas com top 10

N = 10

for target in TARGETS:
    for dataset in SETS:
        
        col_r2 = f"R2_{dataset}_{target}"
        col_mse = f"MSE_{dataset}_{target}"
        
        if col_r2 in results.columns:
            
            table = results[
                ["model", "Neurons", col_r2, col_mse]
            ].sort_values(by=col_r2, ascending=False)
            
            # 🔹 Remove colchetes
            for col in [col_r2, col_mse]:
                table[col] = (
                    table[col]
                    .astype(str)
                    .str.strip("[]")
                    .astype(float)
                )
            
            best_models_tables[f"{dataset}_{target}"] = table
            
            # ===============================
            # 🔹 Estatísticas - TODOS
            # ===============================
            summary_all.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": table[col_r2].mean(),
                "std_r2": table[col_r2].std(),
                "mean_mse": table[col_mse].mean(),
                "std_mse": table[col_mse].std()
            })
            
            # ===============================
            # 🔹 Estatísticas - TOP 10
            # ===============================
            top10 = table.head(N)
            
            summary_top10.append({
                "dataset": dataset,
                "target": target,
                "mean_r2": top10[col_r2].mean(),
                "std_r2": top10[col_r2].std(),
                "mean_mse": top10[col_mse].mean(),
                "std_mse": top10[col_mse].std()
            })


# 🔹 DataFrames finais
df_summary_all = pd.DataFrame(summary_all)
df_summary_top10 = pd.DataFrame(summary_top10)


# 🔹 Exportar para duas abas
with pd.ExcelWriter("Resumo_Estatisticas.xlsx") as writer:
    df_summary_all.to_excel(writer, sheet_name="Todos_Modelos", index=False)
    df_summary_top10.to_excel(writer, sheet_name="Top_10_Modelos", index=False)

print("Arquivo exportado com duas abas com sucesso.")


Arquivo exportado com duas abas com sucesso.


In [9]:
best_only = []
for dataset in SETS:
    for target in TARGETS:
        col_r2 = f"R2_{dataset}_{target}"
        
        if col_r2 in results.columns:
            
            idx_best = results[col_r2].idxmax()
            
            best_only.append({
                "Target": target,
                "Dataset": dataset,
                "Best_Model": results.loc[idx_best, "model"],
                "Neurons": results.loc[idx_best, "Neurons"],
                "Best_R2": results.loc[idx_best, col_r2]
            })

best_only_df = pd.DataFrame(best_only)

print("\n===== MELHOR MODELO POR TARGET/DATASET =====")
display(best_only_df)



===== MELHOR MODELO POR TARGET/DATASET =====


,Target,Dataset,Best_Model,Neurons,Best_R2
0,Y,Train_1,model_128_0,"[264, 128]",[0.9860116118341234]
1,Y,Train_2,model_4_2,"[16, 8, 4]",[0.9811180090287269]
2,Y,Val,model_128_2,[128],[0.9977199879968875]
3,Y,Test_1,model_128_0,"[264, 128]",[0.9848120452766015]
4,Y,Test_2,model_8_3,"[16, 8]",[0.9842225433614167]
5,Y,Test_3,model_8_4,"[32, 16, 8]",[0.9837990796084248]


In [10]:
df_summary_top10["bestR2"] = best_only_df["Best_R2"].str.strip("[]").astype(float)
df_summary_top10["Neurons"] = best_only_df["Neurons"]
display(df_summary_top10)

,dataset,target,mean_r2,std_r2,mean_mse,std_mse,bestR2,Neurons
0,Train_1,Y,0.961419,0.019017,0.003233,0.001594,0.986012,"[264, 128]"
1,Train_2,Y,0.975357,0.008488,0.002654,0.000914,0.981118,"[16, 8, 4]"
2,Val,Y,0.996211,0.000875,0.000282,0.000065,0.997720,[128]
3,Test_1,Y,0.941297,0.025162,0.004811,0.002062,0.984812,"[264, 128]"
4,Test_2,Y,0.970432,0.008354,0.003138,0.000887,0.984223,"[16, 8]"
5,Test_3,Y,0.976620,0.003493,0.001796,0.000268,0.983799,"[32, 16, 8]"


- Entradas: SWdRef, SWeRef, Theta
- Saidas: X
- Trajetorias: ZZ + ZZRoted